In [1]:
import pandas as pd
import os
import openpyxl
import pygwalker as pyg
import numpy as np
import re
import sys
from pathlib import Path

In [2]:
# To add project files
# Keeps going up project structure until it gets to the root.
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69a5ff07-91e0-8329-88d1-cf2ea6c159c2
project_root = Path().resolve()
while project_root.name != "AFOLU_GHG_flux_model":
    project_root = project_root.parent

sys.path.append(str(project_root))

from src.utilities import constants_and_names as cn

In [3]:
# Vegetation zonal stats output
veg_zonal_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/vegetation_v{cn.veg_model_version_underscore}_standard_global__20260224/'
veg_parquet_name = f'veg_model_zonal_stats_v{cn.veg_model_version_underscore}_20260224_19_21_07.parquet'

In [4]:
# Reads vegetation zonal stats parquet table
veg_df_raw = pd.read_parquet(f'{veg_zonal_stats_folder}{veg_parquet_name}')

In [6]:
veg_df_raw

,analysis_layer,adm0,land_state_node,WDPA,cont_eco,Landmark,starting_composite_primary_forest,year,value,tile_id,area_ha,land_state_meaning,land_state_broad_class,land_state_detailed_class,country_name,region,continent,continent_ecozone,WDPA_type,density__Mg_ha
0,gross_emissions__AGC__MgCO2,NA,12100000,0,1020,0,0,2016,8.630256,00N_000E,1.230781,Temporary loss of mangroves,tree,tree_loss,no_country,no_country,Africa,Tropical rainforest,NA,7.012017
1,gross_emissions__AGC__MgCO2,NA,12100000,0,1020,0,0,2017,12.399805,00N_000E,0.230765,Temporary loss of mangroves,tree,tree_loss,no_country,no_country,Africa,Tropical rainforest,NA,53.733429
2,gross_emissions__AGC__MgCO2,NA,12100000,0,1020,0,1,2016,58.504814,00N_000E,1.923030,Temporary loss of mangroves,tree,tree_loss,no_country,no_country,Africa,Tropical rainforest,NA,30.423250
3,gross_emissions__AGC__MgCO2,NA,12100000,0,1020,0,1,2017,16.906231,00N_000E,0.615387,Temporary loss of mangroves,tree,tree_loss,no_country,no_country,Africa,Tropical rainforest,NA,27.472500
4,gross_emissions__AGC__MgCO2,NA,12100000,11,1020,0,0,2016,1.523145,00N_000E,1.538519,Temporary loss of mangroves,tree,tree_loss,no_country,no_country,Africa,Tropical rainforest,Not Reported,0.990007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27999042,net_flux__BGC__MgCO2,USA,62120000,0,2005,1,0,2022,3.692785,80N_170W,0.469234,Short vegetation loss converted to water,short_veg,short_veg_loss,United States of America (the),North America,America North,Polar,NA,7.869821
27999043,gross_emissions__all_C_pools__CO2_only__MgCO2,USA,62220000,0,2005,0,0,2018,0.271070,80N_170W,0.052872,Short vegetation loss converted to non-water w...,short_veg,short_veg_loss,United States of America (the),North America,America North,Polar,NA,5.126917
27999044,gross_emissions__all_C_pools__all_gases__MgCO2e,USA,62220000,0,2005,0,0,2018,0.271070,80N_170W,0.052872,Short vegetation loss converted to non-water w...,short_veg,short_veg_loss,United States of America (the),North America,America North,Polar,NA,5.126917
27999045,net_flux__AGC__MgCO2,USA,62220000,0,2005,0,0,2018,0.054214,80N_170W,0.052872,Short vegetation loss converted to non-water w...,short_veg,short_veg_loss,United States of America (the),North America,America North,Polar,NA,1.025383


In [8]:
### Areas of forest change, annual carbon carbon densities, and annual fluxes for uncertainty analysis

veg_df_uncert = veg_df_raw.copy()

veg_df_area_ts = (
     veg_df_uncert   
    .groupby(["land_state_node", "land_state_meaning", "land_state_broad_class", "land_state_detailed_class", "year", "analysis_layer"], as_index=False)
    .agg({"area_ha": "sum", "value": "sum"})
)
veg_df_area_ts["area_Mha"] = veg_df_area_ts["area_ha"] / 1e6

veg_df_area_ts.to_csv("/mnt/c/GIS/veg_model_area_outputs_for_uncert_analysis.csv", index=False)  # Export area as csv 
veg_df_area_ts

,land_state_node,land_state_meaning,land_state_broad_class,land_state_detailed_class,year,analysis_layer,area_ha,value,area_Mha
0,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,2016,carbon_density__non_soil__MgC_ha,1.620982e+05,1.136252e+06,0.162098
1,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,2016,gross_removals__AGC__MgCO2,1.620982e+05,-2.413498e+06,0.162098
2,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,2016,gross_removals__BGC__MgCO2,1.620982e+05,-1.248027e+06,0.162098
3,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,2016,gross_removals__all_C_pools__MgCO2,1.620982e+05,-4.031480e+06,0.162098
4,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,2016,gross_removals__deadwood_C__MgCO2,1.620982e+05,-3.473215e+05,0.162098
...,...,...,...,...,...,...,...,...,...
7228,70000000,Not in decision tree,no_flux,no_flux,2020,carbon_density__non_soil__MgC_ha,2.357357e+09,1.713759e+06,2357.356445
7229,70000000,Not in decision tree,no_flux,no_flux,2021,carbon_density__non_soil__MgC_ha,2.465766e+09,2.295650e+06,2465.766113
7230,70000000,Not in decision tree,no_flux,no_flux,2022,carbon_density__non_soil__MgC_ha,2.463946e+09,2.464142e+06,2463.946533
7231,70000000,Not in decision tree,no_flux,no_flux,2023,carbon_density__non_soil__MgC_ha,2.450529e+09,2.736400e+06,2450.528564
